# Authentication

Hầu hết các remote MCP server đều yêu cầu xác thực. [`MCPAdapter`](https://reference.langchain.com/python/langchain/mcp/adapter/MCPAdapter) ủy quyền phần xác thực cho FastMCP, nên bất kỳ credential nào mà `fastmcp.Client` chấp nhận đều dùng được: [bearer token](https://docs.langchain.com/oss/python/langchain/mcp/auth#bearer-token) tĩnh, luồng [OAuth 2.1](https://docs.langchain.com/oss/python/langchain/mcp/auth#oauth-authentication) đầy đủ, hoặc bất kỳ [`httpx.Auth`](https://www.python-httpx.org/advanced/authentication/) nào. Hãy truyền credential vào một client dựng sẵn, rồi giao client đó cho adapter. Khi một agent làm việc với nhiều server, hãy dùng [xác thực theo từng server](https://docs.langchain.com/oss/python/langchain/mcp/auth#per-server-authentication); khi triển khai, hãy dùng [xác thực theo từng người dùng](https://docs.langchain.com/oss/python/langchain/mcp/auth#per-user-authentication) để mỗi lần chạy truy cập server với tư cách của người gọi.

<div class="alert alert-info">

Namespace `langchain.mcp` yêu cầu `langchain[mcp]>=1.4.0` và đang ở giai đoạn beta. API có thể thay đổi.

</div>

## Bearer token

Trường hợp đơn giản nhất: server xác minh một token mà bạn đã cấp phát, và token này được đưa vào header `Authorization: Bearer <token>`. Không có bước discovery, không cần trình duyệt và không có refresh. Hãy truyền token vào tham số `auth` của client:

In [ ]:
from fastmcp.client import Client
from langchain.mcp import MCPAdapter


async def load_tools_with_bearer(url: str, token: str) -> list:
    # `auth` nhận một chuỗi bearer token, chuỗi ký tự "oauth" (luồng OAuth 2.1
    # đầy đủ với dynamic client registration), hoặc bất kỳ đối tượng `httpx.Auth` nào.
    async with MCPAdapter(Client(url, auth=token)) as adapter:
        return await adapter.list_tools()

Tham số `auth` nhận một chuỗi bearer token, chuỗi ký tự `"oauth"`, hoặc bất kỳ `httpx.Auth` nào. Trong một fleet `MCPConfig`, mỗi server dùng cùng một khóa này.

## Xác thực OAuth

Với server tự cấp credential, hãy truyền chuỗi ký tự `"oauth"`. FastMCP sẽ chạy toàn bộ luồng OAuth 2.1: discovery, [dynamic client registration](https://modelcontextprotocol.io/specification/draft/basic/authorization), chuyển hướng qua trình duyệt và trao đổi token. Dynamic client registration nghĩa là client tự đăng ký tại thời điểm chạy, thay vì bạn phải cấp sẵn client ID:

In [ ]:
async def load_tools_with_oauth(url: str) -> list:
    # "oauth" sẽ chạy discovery, dynamic client registration, chuyển hướng qua
    # trình duyệt và trao đổi token. Truyền `OAuth(..., token_storage=...)` để lưu
    # token qua nhiều lần chạy, thay vì lặp lại bước mở trình duyệt mỗi lần.
    async with MCPAdapter(Client(url, auth="oauth")) as adapter:
        return await adapter.list_tools()

Theo mặc định, token được giữ trong bộ nhớ, nên mỗi lần chạy đều phải lặp lại bước mở trình duyệt. Hãy truyền một provider `OAuth` dựng sẵn cùng với token store để lưu token qua nhiều lần chạy:

In [ ]:
from fastmcp.client import Client
from fastmcp.client.auth import OAuth

oauth = OAuth(mcp_url="https://example.com/mcp", token_storage=...)
client = Client("https://example.com/mcp", auth=oauth)

FastMCP cung cấp sẵn các provider cho những identity provider phổ biến (Auth0, WorkOS, Okta, v.v.); luồng mà client thực thi là như nhau ở tất cả các provider này. Xem [Xác thực OAuth](https://gofastmcp.com/clients/auth/oauth) trong tài liệu FastMCP.

## Xác thực theo từng server

Khi một agent làm việc với nhiều server, mỗi server có thể cần credential riêng. Hãy tạo một kết nối riêng cho từng server bằng [`ClientGroup`](https://docs.langchain.com/oss/python/langchain/mcp/connections#independent-connections-with-clientgroup) và thiết lập `auth` cho từng client, để mỗi server được xác thực độc lập:

In [ ]:
from fastmcp.client.group import ClientGroup


async def load_with_per_server_auth(
    billing_url: str, docs_token: str, docs_url: str
) -> list:
    # Mỗi server mang credential riêng của nó. `ClientGroup` giữ một kết nối
    # cho mỗi server, nên từng server được xác thực độc lập.
    group = ClientGroup(
        {
            "billing": Client(billing_url, auth="oauth"),
            "docs": Client(docs_url, auth=docs_token),
        }
    )
    async with MCPAdapter(group) as adapter:
        return await adapter.list_tools()

## Xác thực theo từng người dùng

Trong môi trường triển khai, mỗi lần chạy nên truy cập MCP server với tư cách của chính người dùng đã khởi tạo nó, thay vì dùng chung một credential. Mẫu thiết kế này gồm hai phần:

1. **Xác thực người gọi tại LangGraph server.** Một [custom auth handler](https://docs.langchain.com/langsmith/auth) phân giải request đến thành danh tính người dùng, và mỗi lần chạy sẽ đọc danh tính này từ runtime của nó.
2. **Tạo hoặc đổi credential cho người dùng đó.** Bên trong [graph factory](https://docs.langchain.com/oss/python/langchain/mcp/connections#scale-a-deployment), đọc danh tính của người dùng và dựng MCP client với token riêng cho từng người dùng, để kết nối mang theo quyền hạn của chính người dùng đó.

In [ ]:
from fastmcp.client import Client
from fastmcp.client.auth import BearerAuth

CONFIG = {
    "mcpServers": {
        "docs": { "url": "https://example.com/mcp" }
    }
}

async def make_graph(runtime):
    user = runtime.user.identity if runtime.user is not None else "anonymous"
    auth = BearerAuth(token_for(user))  # đổi lấy token riêng cho từng người dùng
    async with MCPAdapter(Client(CONFIG, auth=auth)) as adapter:
        tools = await adapter.list_tools()
        return create_agent("claude-sonnet-5", tools)

Trong môi trường production, `token_for` đại diện cho bất kỳ giải pháp nào mà hệ thống triển khai của bạn đã có: một OAuth gateway đổi session lấy token riêng cho từng người dùng, hoặc một provider trong `fastmcp.client.auth` chạy luồng authorization-code cho từng danh tính. Hãy cô lập mọi [phản hồi được cache](https://docs.langchain.com/oss/python/langchain/mcp/connections#caching) theo từng người dùng, dùng danh tính đã được xác minh làm khóa, để không người dùng nào nhìn thấy danh sách tool đã cache của người khác.

## Xem thêm

* [Xác thực OAuth của FastMCP](https://gofastmcp.com/clients/auth/oauth)
* [Xác thực bằng bearer token của FastMCP](https://gofastmcp.com/clients/auth/bearer)
* [Xác thực machine-to-machine của FastMCP](https://gofastmcp.com/clients/auth/client-credentials)
* [Client group của FastMCP](https://gofastmcp.com/clients/client-groups) — các kết nối độc lập cho xác thực theo từng server
* [Các authentication provider phía server của FastMCP](https://gofastmcp.com/servers/auth/authentication)
* [Đặc tả MCP authorization](https://modelcontextprotocol.io/specification/draft/basic/authorization)
* [Xác thực tùy chỉnh cho LangGraph server](https://docs.langchain.com/langsmith/auth)